# 07 - Feature Selection Pix

Seleciona features candidatas para modelos educacionais de regressão e classificação.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name in {"notebooks", "i_notebooks"} else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.config import FIGURES_DIR, PIX_ML_FEATURES_DIR, PIX_SELECTED_FEATURES_DIR, REPORTS_DIR, create_project_directories
from src.data_quality import ensure_not_empty
from src.spark_session import get_spark_session

create_project_directories(False)
spark = get_spark_session("07-feature-selection-pix")

In [ ]:
features_df = spark.read.parquet(str(PIX_ML_FEATURES_DIR))
ensure_not_empty(features_df, "Gold features Pix")
selected_features = [
    "mes_numero", "trimestre", "valor_total_lag_1", "quantidade_transacoes_lag_1",
    "ticket_medio_lag_1", "crescimento_valor_lag_1", "crescimento_qtd_lag_1",
    "valor_total_mm3", "quantidade_transacoes_mm3", "ticket_medio_mm3",
    "flag_crescimento_valor", "flag_crescimento_qtd"
]
selection_rows = [(feature, "selecionada", "Feature temporal ou estatística derivada sem identificar diretamente o mês alvo.") for feature in selected_features]
selection_df = spark.createDataFrame(selection_rows, ["feature", "status", "criterio"])
selection_df.write.mode("overwrite").parquet(str(PIX_SELECTED_FEATURES_DIR))
selection_df.toPandas().to_csv(REPORTS_DIR / "feature_selection_summary.csv", index=False)
selection_df.show(truncate=False)

In [ ]:
pd_features = features_df.select(selected_features).toPandas().fillna(0)
corr = pd_features.corr(numeric_only=True)
corr.to_csv(REPORTS_DIR / "feature_correlation_matrix.csv")
plt.figure(figsize=(11, 9))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlação")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlação entre features selecionadas")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "10_pix_feature_correlation_heatmap.png", dpi=160)
plt.close()

In [ ]:
spark.stop()